In [ ]:
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go 
import matplotlib.pyplot as plt

In [ ]:
renderer = "browser"

In [ ]:
# il ds su disco è di circa 400 kb 
bqdata = pd.read_csv("./bq-data-export.csv")
bqdata.head(10)

In [ ]:
fig = px.line(bqdata, x = 'record_time', y = 'duration_in_seconds')
fig.update_traces(name='Real Duration', showlegend=True)
fig.add_scatter(
    x=bqdata['record_time'], 
    y=bqdata['static_duration_in_seconds'], 
    name='Static Duration',               # Il nome che apparirà nella legenda
    line=dict(color='orange'))           #)
fig.update_xaxes(tickangle=45,nticks=10,)
# fig.update_layout(
#         width=1200,  # Larghezza 
#         height=700   # Altezza 
#     )
fig.show(renderer=renderer)

bqdata = bqdata.drop_duplicates()

In [ ]:
if not os.path.exists("bq-data-export.csv"):
    print(f"Errore: Il file '{"bq-data-export.csv"}' non è stato trovato. Verifica il percorso.")
else:
    # Calcola la dimensione del file di test (1 strada, 1 settimana)
    size_bytes = os.path.getsize("bq-data-export.csv")
    size_mb = size_bytes / (1024 ** 2)
    
    print("=" * 50)
    print(f"ANALISI FILE DI TEST (1 Strada, 1 Settimana)")
    print("=" * 50)
    print(f"Dimensione effettiva: {size_mb:.2f} MB")
    
    try:
        df = pd.read_csv("bq-data-export.csv")
        print(f"Numero totale di righe campionate: {len(df):,}")
        
        # if 'record_time' in df.columns:
        #     df['record_time'] = pd.to_datetime(df['record_time'])
        #     df = df.sort_values('record_time')
        #     frequenza = df['record_time'].diff().mode()[0]
        #     print(f"Frequenza di aggiornamento dei dati: ogni {frequenza}")
    except Exception as e:
        print(f"Nota: Impossibile leggere il contenuto del file per dettagli extra ({e})")

    # 3. Estrapolazione su 1 anno e 30 strade
    settimane_anno = 52
    numero_strade = 30
    
    dimensione_stimata_mb = size_mb * settimane_anno * numero_strade
    dimensione_stimata_gb = dimensione_stimata_mb / 1024
    
    print("\n" + "=" * 50)
    print(f"PROIEZIONE SU 1 ANNO E 30 STRADE")
    print("=" * 50)
    print(f"Dimensione stimata totale in Megabyte: {dimensione_stimata_mb:.2f} MB")
    print(f"Dimensione stimata totale in Gigabyte: {dimensione_stimata_gb:.2f} GB")
    
    # Determina l'ordine di grandezza complessivo
    if dimensione_stimata_gb < 1:
        print("\n--> Ordine di grandezza previsto: Megabytes (MB)")
    elif dimensione_stimata_gb < 1024:
        print("\n--> Ordine di grandezza previsto: Gigabytes (GB)")
    else:
        print("\n--> Ordine di grandezza previsto: Terabytes (TB)")
    print("=" * 50)

In [ ]:
# 1. Convertiamo in datetime vero e rimuoviamo il fuso orario UTC (diventa datetime64 senza stringhe)
bqdata['record_time'] = pd.to_datetime(bqdata['record_time']).dt.tz_localize(None)
df = bqdata.sort_values('record_time')
df['time_delta'] = df['record_time'].diff()
print(df['record_time'].dtype)

In [ ]:
fig = px.line(bqdata, x = 'record_time', y = 'duration_in_seconds')
fig.update_traces(name='Real Duration', showlegend=True)
fig.add_scatter(
    x=bqdata['record_time'], 
    y=bqdata['static_duration_in_seconds'], 
    name='Static Duration',               # Il nome che apparirà nella legenda
    line=dict(color='orange'))           #)
fig.update_xaxes(tickangle=45,nticks=10,)
fig.update_layout(
        width=1200,  # Larghezza 
        height=700   # Altezza 
    )
fig.show(renderer=renderer)

In [ ]:
df['secondi_dt'] = df['time_delta'].dt.total_seconds() / 60
fig, ax = plt.subplots(figsize=(8, 5))
df['secondi_dt'].hist(bins=50, edgecolor='black', ax=ax, color='skyblue')

ax.set_title('Distribution', fontsize=14)
ax.set_xlabel('Minutes between records', fontsize=12)
ax.set_ylabel('Number of occurences', fontsize=12)
ax.grid(axis='y', alpha=0.75)
plt.savefig('Hist_minutes.png', bbox_inches='tight')
# Se sei su Jupyter Notebook puoi usare: plt.show()
df

In [ ]:
# df[(df['secondi_dt'].round() != 2) & (df['secondi_dt'].notna())]
df[(df['record_time'].dt.day == 16) & (df['record_time'].dt.hour > 13) ].tail(30)

In [ ]:
def plot_sovrapposto_con_media(df, col_data='record_time', col_valore='duration_in_seconds'):
    """
    Prende un DataFrame, sovrappone i grafici giornalieri (00:00 - 24:00) 
    e mostra la linea della media in evidenza con legenda raggruppata.
    """
    # Consiglio: usiamo .copy() per evitare di modificare il df originale fuori dalla funzione
    df = df.copy()
    
    df['day'] = df[col_data].dt.strftime('%Y-%m-%d')    
    df['hour_minute'] = df[col_data].dt.round('min').dt.strftime('%H:%M')    
    
    # Calcoliamo la media per ogni timestamp (hour_minute)
    mean_df = df.groupby('hour_minute')[col_valore].mean().reset_index()
    mean_df = mean_df.sort_values('hour_minute')
    
    fig = go.Figure()
    
    days = df['day'].unique()
    for i, day in enumerate(days):
        df_day = df[df['day'] == day].sort_values('hour_minute')

        # Mostriamo la voce in legenda SOLO al primo giro del ciclo (quando i == 0)
        mostra_nella_legenda = True if i == 0 else False

        # 1. Linee leggere del valore REALE (es. duration_in_seconds)
        fig.add_trace(go.Scatter(
            x=df_day['hour_minute'],
            y=df_day[col_valore],
            mode='lines',
            name='Duration',         # Nome unico per la legenda
            legendgroup='group_overlaped',   # Stesso gruppo per tutte le linee reali
            showlegend=mostra_nella_legenda,      # Compare solo una volta in legenda
            line=dict(color='rgba(31, 119, 180, 0.3)', width=1), # Abbassata opacità a 0.2 per renderle più "leggere"
        ))
        
        # 2. Linee leggere del valore STATIC
        fig.add_trace(go.Scatter(
            x=df_day['hour_minute'],
            y=df_day['static_duration_in_seconds'],
            mode='lines',
            name='Static duration',       # Nome unico per la legenda
            legendgroup='group_static',    # Stesso gruppo per tutte le linee statiche
            showlegend=mostra_nella_legenda,      # Compare solo una volta in legenda
            line=dict(color='rgba(251, 141, 180, 0.3)', width=1),
        ))
    
    # 3. Linea della MEDIA (essendo fuori dal ciclo è singola, mostra la legenda di default)
    fig.add_trace(go.Scatter(
        x=mean_df['hour_minute'],
        y=mean_df[col_valore],
        mode='lines',
        name='Mean',
        line=dict(color='rgba(31, 119, 180, 1.0)', width=3), # Più spessa e accesa
    ))
    
    fig.update_layout(
        title=f"Analisi Andamento: {col_valore}",
        xaxis_title="Hour of the day (HH:MM)",
        yaxis_title=col_valore,
        xaxis={'categoryorder': 'category ascending'}, 
        hovermode='x unified', 
        template='plotly_white'
    )
    # fig.update_layout(
    #     width=1200,  # Larghezza 
    #     height=700   # Altezza 
    # )
    fig.show(renderer=renderer)

# Esegui la funzione
plot_sovrapposto_con_media(df)

In [ ]:
df['time_delta_rounded'] = df['time_delta'].dt.round('min')
lends = df.shape[0]
print("perc. dataset con durata 942:", len(df[df["static_duration_in_seconds"] == 942]) / lends * 100 , "%")
print("perc. dataset con durata 945", len(df[df["static_duration_in_seconds"] == 945]) / lends * 100 , "%")

In [ ]:
# 1. Convertiamo il time_delta in secondi totali (restituisce un numero float, es. 121.02)
df['secondi_delta'] = df['time_delta'].dt.total_seconds()

# 2. Creiamo una condizione: cerchiamo dove i secondi sono FUORI dall'intervallo [118, 122]
# Escludiamo anche i valori NaT (la prima riga)

anomalie_tolleranza = df[((df['secondi_delta'] < 116) | (df['secondi_delta'] > 124)) & (df['secondi_delta'].notna())]

# 3. Mostriamo i risultati
print(f"Numero di righe fuori tolleranza: {len(anomalie_tolleranza)}")
if not anomalie_tolleranza.empty:
    print(anomalie_tolleranza[['record_time', 'time_delta', 'secondi_delta']].head())